# Multi-Format Orthorectification Comparison

**Consolidates**: `ortho_combined.ipynb` + `compare_sidd_ortho.ipynb`  
**Source**: `grdl/example/ortho/`

## Learning Objectives

- Compare orthorectification approaches across SAR formats (SICD, SIDD, BIOMASS)
- Understand geolocation backend differences (R/Rdot vs plane projection vs GCP)
- Evaluate SIDD producer projection vs custom re-orthorectification
- Measure timing and accuracy metrics for each approach

## Theory: Format-Specific Geolocation

| Format | Geolocation Method | Backend |
|--------|-------------------|----------|
| **SICD** | R/Rdot inverse iteration | Native GRDL or sarpy/sarkit |
| **SIDD** | Plane projection inverse | Native GRDL (RPC-style polynomial) |
| **BIOMASS** | GCP Delaunay triangulation | Scipy |

**SICD**: Most accurate for slant-range products. Requires iterative solver for terrain correction.  
**SIDD**: Pre-projected by producer. Re-ortho useful for custom grids or DEM updates.  
**BIOMASS**: Coarse GCP grid (L1 products lack dense metadata). Best-effort geolocation.

## Data Requirements

This notebook is **multi-format** — you can run with any combination of:
- SICD file (`.nitf`, `.sicd`)
- SIDD file (`.nitf`, `.sidd`)
- BIOMASS product directory

Configure paths in the Configuration cell. Missing formats will be skipped automatically.

## Imports & Environment Validation

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
from pathlib import Path
import time
import gc
import numpy as np
import napari

from grdl.IO import open_reader
from grdl.geolocation import create_geolocation
from grdl.geolocation.chip import ChipGeolocation
from grdl.image_processing.ortho import orthorectify, UTMGrid

## Configuration

In [ ]:
# File paths (configure at least one)
SICD_PATH = Path("/path/to/your/sicd_file.nitf")  # Optional
SIDD_PATH = Path("/path/to/your/sidd_file.nitf")  # Optional
BIOMASS_PATH = Path("/path/to/your/biomass_product")  # Optional

# Processing parameters
CHIP_SIZE = 2048  # Pixels (will be reduced if image is smaller)
OUTPUT_PIXEL_SPACING = 0.5  # meters (UTM grid resolution)
INTERPOLATION = 'bilinear'  # Options: 'nearest', 'bilinear', 'bicubic'

# Collect available formats
files_to_compare = []
if SICD_PATH.exists():
    files_to_compare.append(('SICD', SICD_PATH))
if SIDD_PATH.exists():
    files_to_compare.append(('SIDD', SIDD_PATH))
if BIOMASS_PATH.exists():
    files_to_compare.append(('BIOMASS', BIOMASS_PATH))

if len(files_to_compare) == 0:
    raise FileNotFoundError("No valid files found. Configure at least one path above.")

print(f"Formats to compare: {[fmt for fmt, _ in files_to_compare]}")
print(f"Chip size: {CHIP_SIZE} px")
print(f"Output resolution: {OUTPUT_PIXEL_SPACING} m")
print(f"Interpolation: {INTERPOLATION}")

## Part 1: Multi-Format Comparison (SICD vs SIDD vs BIOMASS)

Extract center chips from each format and orthorectify to a common UTM grid. Compare slant-range vs orthorectified outputs side-by-side.

In [ ]:
ortho_results = []
timing_metrics = []

for fmt, path in files_to_compare:
    print(f"\nProcessing {fmt}...")
    print("=" * 60)
    
    # CRITICAL: Use context manager to prevent file locks
    with open_reader(path) as reader:
        meta = reader.metadata
        rows, cols = reader.shape[:2]
        
        print(f"  Image size: {rows} × {cols} px")
        
        # Compute chip region (center extract)
        actual_chip_size = min(CHIP_SIZE, rows, cols)
        r_start = (rows - actual_chip_size) // 2
        c_start = (cols - actual_chip_size) // 2
        
        print(f"  Extracting chip: [{r_start}:{r_start+actual_chip_size}, {c_start}:{c_start+actual_chip_size}]")
        
        # Read chip
        chip = reader.read(
            row_start=r_start,
            row_end=r_start + actual_chip_size,
            col_start=c_start,
            col_end=c_start + actual_chip_size,
        )
        
        # Convert complex to magnitude
        if np.iscomplexobj(chip):
            chip = np.abs(chip)
            print(f"  Converted complex → magnitude")
        
        # Create geolocation with chip offset
        geo_full = create_geolocation(reader)
        geo_chip = ChipGeolocation(geo_full, row_offset=r_start, col_offset=c_start)
        
        # Define output grid (UTM centered on chip center)
        center_lat, center_lon, _ = geo_chip.image_to_latlon(actual_chip_size // 2, actual_chip_size // 2)
        extent_m = actual_chip_size * OUTPUT_PIXEL_SPACING
        output_grid = UTMGrid.from_center(
            center_lat,
            center_lon,
            east_extent_m=extent_m / 2,
            north_extent_m=extent_m / 2,
            pixel_spacing_m=OUTPUT_PIXEL_SPACING,
        )
        
        print(f"  Output grid: {output_grid.shape[0]} × {output_grid.shape[1]} px (UTM zone {output_grid.zone}{output_grid.hemisphere})")
        
        # Orthorectify (timed)
        t0 = time.time()
        ortho_result = orthorectify(
            source=chip,
            geolocation=geo_chip,
            output_grid=output_grid,
            interpolation=INTERPOLATION,
        )
        elapsed = time.time() - t0
        
        ortho = ortho_result.data
        
        print(f"  ✓ Orthorectified in {elapsed:.2f}s")
        print(f"    Output shape: {ortho.shape}")
        print(f"    Valid pixels: {np.sum(~np.isnan(ortho))} / {ortho.size} ({100*np.sum(~np.isnan(ortho))/ortho.size:.1f}%)")
        
        # Store results
        ortho_results.append((fmt, chip, ortho))
        timing_metrics.append((fmt, elapsed, ortho.shape))

# Reader context automatically closed after each iteration

print("\n" + "=" * 60)
print("✓ All formats processed")

### Timing & Quality Metrics

In [ ]:
print("\nOrthorectification Performance Summary")
print("=" * 80)
print(f"{'Format':<10} {'Time (s)':<12} {'Output Shape':<20} {'Pixels/sec':<15}")
print("=" * 80)

for fmt, elapsed, shape in timing_metrics:
    total_pixels = shape[0] * shape[1]
    pixels_per_sec = total_pixels / elapsed
    print(f"{fmt:<10} {elapsed:<12.2f} {str(shape):<20} {pixels_per_sec:<15.0f}")

print("=" * 80)

## Part 2: SIDD Producer Projection vs Custom Re-Ortho

**SIDD-specific analysis**: Compare the producer's native projection (e.g., Geographic, UTM, Plane) against custom re-orthorectification to a different grid. This reveals:
- Projection accuracy from producer
- Impact of DEM updates
- Geometric distortion from grid misalignment

**Note**: This section only runs if SIDD is available.

In [ ]:
sidd_comparison_results = None

if SIDD_PATH.exists():
    print("\nSIDD Producer Projection Analysis")
    print("=" * 60)
    
    # CRITICAL: Use context manager
    with open_reader(SIDD_PATH) as reader:
        meta = reader.metadata
        
        # Inspect producer projection
        try:
            proj_type = meta.measurement.plane_projection.projection_type
            print(f"  Producer projection type: {proj_type}")
        except AttributeError:
            proj_type = "Unknown"
            print(f"  Producer projection type: {proj_type} (metadata unavailable)")
        
        rows, cols = reader.shape[:2]
        actual_chip_size = min(CHIP_SIZE, rows, cols)
        r_start = (rows - actual_chip_size) // 2
        c_start = (cols - actual_chip_size) // 2
        
        # Read original SIDD (already projected by producer)
        original_sidd = reader.read(
            row_start=r_start,
            row_end=r_start + actual_chip_size,
            col_start=c_start,
            col_end=c_start + actual_chip_size,
        )
        
        # Re-orthorectify to custom UTM grid
        geo_full = create_geolocation(reader)
        geo_chip = ChipGeolocation(geo_full, row_offset=r_start, col_offset=c_start)
        
        center_lat, center_lon, _ = geo_chip.image_to_latlon(actual_chip_size // 2, actual_chip_size // 2)
        extent_m = actual_chip_size * OUTPUT_PIXEL_SPACING
        output_grid = UTMGrid.from_center(
            center_lat,
            center_lon,
            east_extent_m=extent_m / 2,
            north_extent_m=extent_m / 2,
            pixel_spacing_m=OUTPUT_PIXEL_SPACING,
        )
        
        re_ortho = orthorectify(
            source=original_sidd,
            geolocation=geo_chip,
            output_grid=output_grid,
            interpolation=INTERPOLATION,
        ).data
    
    # Compute difference map (resize original to match re-ortho for pixel-wise comparison)
    from scipy.ndimage import zoom
    scale = (re_ortho.shape[0] / original_sidd.shape[0], re_ortho.shape[1] / original_sidd.shape[1])
    original_resized = zoom(original_sidd, scale, order=1)
    
    difference = np.abs(re_ortho - original_resized)
    
    # Compute statistics on difference
    valid_mask = ~np.isnan(difference)
    mean_diff = np.nanmean(difference)
    max_diff = np.nanmax(difference)
    rmse = np.sqrt(np.nanmean(difference[valid_mask]**2))
    
    print(f"\n  Difference Statistics (Producer vs Re-Ortho):")
    print(f"    Mean absolute error: {mean_diff:.3f}")
    print(f"    Max absolute error: {max_diff:.3f}")
    print(f"    RMSE: {rmse:.3f}")
    print(f"    Valid comparison pixels: {np.sum(valid_mask)} / {difference.size}")
    
    sidd_comparison_results = (original_sidd, re_ortho, difference, proj_type)
    print("\n  ✓ SIDD comparison complete")
else:
    print("\n⚠ SIDD file not provided — skipping SIDD-specific comparison")

## Visualization — Side-by-Side Multi-Format Comparison

In [ ]:
viewer = napari.Viewer(title="Multi-Format Ortho Comparison")

# Add slant/ortho pairs for each format
for i, (fmt, chip, ortho) in enumerate(ortho_results):
    # Convert to dB for display
    chip_db = 20 * np.log10(np.clip(chip, 1e-12, None))
    ortho_db = 20 * np.log10(np.clip(ortho, 1e-12, None))
    
    # Add slant (visible only for first format)
    viewer.add_image(
        chip_db,
        name=f"{fmt} Slant",
        colormap="gray",
        visible=(i == 0),
    )
    
    # Add ortho (all visible)
    viewer.add_image(
        ortho_db,
        name=f"{fmt} Ortho",
        colormap="gray",
        visible=True,
    )

# Add SIDD comparison if available
if sidd_comparison_results is not None:
    original_sidd, re_ortho, difference, proj_type = sidd_comparison_results
    
    viewer.add_image(
        original_sidd,
        name=f"SIDD Original ({proj_type})",
        colormap="gray",
        visible=False,
    )
    
    viewer.add_image(
        re_ortho,
        name="SIDD Re-Ortho (Custom UTM)",
        colormap="gray",
        visible=False,
    )
    
    viewer.add_image(
        difference,
        name="SIDD Difference Map",
        colormap="inferno",
        visible=False,
    )

# Simplify viewer UI
try:
    viewer.window._qt_viewer.dockLayerControls.setVisible(False)
    viewer.window._qt_viewer.dockConsole.setVisible(False)
    viewer.window._qt_viewer.activityDock.setVisible(False)
except AttributeError:
    pass  # Some docks may not exist in all napari versions

print("\n✓ Napari viewer opened")
print(f"  Formats displayed: {[fmt for fmt, _, _ in ortho_results]}")
print("\nViewer tips:")
print("  - Toggle layer visibility to compare formats")
print("  - Use SIDD Difference Map to see producer vs custom projection errors")
print("\nExecution paused — close the napari window to continue and free memory...")

napari.run()

print("\n✓ Viewer closed — cleaning up memory...")
del ortho_results, sidd_comparison_results, viewer
gc.collect()
print("✓ Memory cleanup complete")

## Summary

**GRDL patterns demonstrated**:
- ✅ Multi-format orthorectification: SICD (R/Rdot), SIDD (plane projection), BIOMASS (GCP)
- ✅ `create_geolocation(reader)` — auto-detect geolocation backend from metadata
- ✅ `ChipGeolocation` — preserve geolocation accuracy for sub-image extracts
- ✅ `UTMGrid.from_center()` — define common output grid for cross-format comparison
- ✅ `orthorectify()` — unified orthorectification interface for all formats
- ✅ SIDD producer projection vs custom re-ortho comparison
- ✅ **Context manager usage** (`with open_reader(path)`) — prevent OS-level file locks

**Key insights**:
- SICD R/Rdot provides highest accuracy for slant-range products
- SIDD producer projection is fast but may not match updated DEMs
- BIOMASS GCP geolocation is approximate (L1 products lack dense metadata)
- Re-orthorectifying SIDD to custom grids allows DEM updates and projection changes

**Next steps**:
- Add DEM terrain correction: see `ortho/ortho_sicd.ipynb`
- Extract specific ROIs: see `ortho/roi_extraction_and_ortho.ipynb`
- Detect targets in orthorectified imagery: see `image_processing/csi_detection_overlay.ipynb`